# PyTorch

## torch模块
torch模块 torch.nn.module 是PyTorch的核心模块，提供了张量（Tensor）操作、自动求导（Autograd）功能以及神经网络（nn）模块等。它是PyTorch中最基础的模块，几乎所有的PyTorch代码都会使用到torch模块。

### PyTorch模型核心术语
包括 `nn.Parameter`, `register_buffer`, `model.parameters`, `model.state_dicr()`, `model.to(device)`等

`nn.Parameter` （可学习参数）
> 模型中需要通过反向传播更新的张量。例如线性层的权重 $W$ 和偏置 $b$。
```python
self.weight = nn.Parameter(torch.randn(3, 3))  # 会被优化器更新
```
创建为 `nn.Parameter` 后，PyTorch 自动：

标记 requires_grad=True（开启梯度追踪），纳入 model.parameters() 列表

而在梯度计算时，如
```python
loss = criterion(model(x), y)   # 前向传播，计算损失
loss.backward()                  # 反向传播，计算每个 Parameter 的梯度 ∂L/∂W
optimizer.step()                 # 用梯度更新参数
```
nn.Parameter → 参与这个过程，会被 backward() 计算梯度，被 optimizer.step() 更新，register_buffer → 不参与，梯度不会流过它，优化器也不碰它。

`model.parameters`（参数迭代器）
> 返回模型中所有 nn.Parameter 的迭代器。最主要的用途是传给优化器：
```python
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
#                            ^^^^^^^^^^^^^^^^
#                            告诉优化器：这些张量需要更新
```
register_buffer 注册的张量不在其中，所以优化器不会去动它们

`model.state_dict()`（模型状态字典）
> 模型的完整快照，是一个 {名称: 张量} 的有序字典。用于保存和加载模型：
```python
# 保存
torch.save(model.state_dict(), "checkpoint.pth")
# 加载
model.load_state_dict(torch.load("checkpoint.pth"))
```
它包含所有 nn.Parameter（权重、偏置等），所有 register_buffer（如 $\beta_t$ 调度序列）

`model.to(device)` (设备迁移)
> 把模型所有张量从 CPU 移到 GPU（或反过来）：
```python
model = model.to("cuda")  # 或 model.cuda()
```
这一步会移动：所有 nn.Parameter 和所有 register_buffer
不会移动普通属性 self.xxx = some_tensor。如果忘了用 register_buffer，训练时就会遇到经典报错：

> RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

**小结：**
```bash
model
├── nn.Parameter (权重W, 偏置b, ...)
│   ├── 有梯度 ✓
│   ├── optimizer 更新 ✓
│   ├── state_dict 包含 ✓
│   └── .to(device) 跟随 ✓
│
├── register_buffer (betas, alpha_bars, ...)
│   ├── 无梯度 ✗
│   ├── optimizer 不碰 ✗
│   ├── state_dict 包含 ✓    ← 会被保存/加载
│   └── .to(device) 跟随 ✓    ← 自动移设备
│
└── 普通属性 (hidden_dim=256, mode="train", ...)
    ├── 无梯度 ✗
    ├── state_dict 不包含 ✗
    └── .to(device) 不跟随 ✗
```

#### register_buffer补充
方法签名如下：
```python
(method) def register_buffer(
    name: str,  # name of the buffer
    tensor: Tensor | None,
    persistent: bool = True  # false则不会保存到model的state_dict
) -> None
```

## torch grad

## 张量操作

### expand和reshape
修改批次样本的尺度维度有用，以适应神经网络的输入维度要求

```python    
def expand_tensor(self, x: torch.Tensor) -> torch.Tensor:
        """
        将张量扩展以支持多时间步训练
        
        训练时，每个batch样本会采样 N_t_training 个不同的扩散时间步，
        该函数将输入张量在时间步维度上复制扩展，然后展平为大batch。
        
        Args:
            x: 输入张量 [B, N, D]
                - B: batch size
                - N: 节点数量
                - D: 特征维度
        
        Returns:
            扩展后的张量 [B*N_t_training, N, D]
            
        示例:
            输入: [2, 10, 64] (2个样本, 10个节点, 64维特征)
            N_t_training = 3
            输出: [6, 10, 64] (每个样本复制3份，共6个样本)
        """
        B, N, _ = x.shape
        return (
            x[:, None, :, :]                              # [B, 1, N, D] - 在时间步维度插入新轴
            .expand(-1, self.N_t_training, -1, -1)        # [B, T, N, D] - 沿时间步维度复制
            .reshape(B * self.N_t_training, N, -1)        # [B*T, N, D] - 展平batch和时间步维度
        )
```

### 广播
标量乘矢量

```python
        noise_V_R_rot = torch.randn_like(V_R_rot)                          # ε_rot ~ N(0,I) [B*T, L, 3]
        a_bar = self.alpha_bars[t][:, None, None]                          # ᾱ_t [B*T, 1, 1] β与t线性 → ᾱ与t指数衰减
        
        # x_t = √ᾱ_t · x_0 + √(1-ᾱ_t) · ε
        # 运算方式: 逐元素乘法(*) + 广播，不是矩阵乘法(@)
        #   a_bar.sqrt()  : [B*T, 1, 1]  — 每个样本一个标量(噪声调度系数)
        #   V_R_trans      : [B*T, L, 3]  — 每个样本 L 个 link 的 3D 坐标
        #   广播机制: [B*T, 1, 1] → [B*T, L, 3]，即同一样本内所有 link、所有坐标维度共享同一缩放系数
        #   等价于: 对每个样本 i, x_t^(i) = scalar_i * matrix_i (标量乘矩阵)
        noisy_V_R_trans = a_bar.sqrt() * V_R_trans + (1 - a_bar).sqrt() * noise_V_R_trans   # x_t 平移 [B*T, L, 3]
        noisy_V_R_rot = a_bar.sqrt() * V_R_rot + (1 - a_bar).sqrt() * noise_V_R_rot         # x_t 旋转 [B*T, L, 3]
```

## 算子操作

### 累乘
torch.prod 是“累乘”操作。它会对指定张量的所有元素进行连乘（即乘积），返回一个标量或张量。例如：
torch.prod(torch.tensor([2, 3, 4])) 的结果是 2 × 3 × 4 = 24

```python
self.register_buffer(
    "alpha_bars",
    torch.tensor([torch.prod(self.alphas[: i + 1]) for i in range(len(self.alphas))]),  # 累乘
)
```